In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from torchvision import transforms
import torchvision

import matplotlib.pyplot as plt

from collections import namedtuple

from sklearn.metrics import classification_report

In [3]:
def get_classes():
  classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
  return classes

TrainTest = namedtuple('TrainTest', ['train', 'test'])

def prepare_data():
  transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
  ])
  transform_test = transforms.Compose([
    transforms.ToTensor()
  ])
  trainset = torchvision.datasets.CIFAR10(root='./data', download=True, train=True, transform=transform_train)
  testset = torchvision.datasets.CIFAR10(root='./data', download=True, train=False, transform=transform_test)
  return TrainTest(train=trainset, test=testset)

def prepare_loader(datasets):
  # On Windows notebooks set num_workers=0 to avoid worker spawn issues
  trainloader = DataLoader(dataset=datasets.train, batch_size=128, shuffle=True, num_workers=0)
  testloader = DataLoader(dataset=datasets.test, batch_size=128, shuffle=False, num_workers=0)
  return TrainTest(train=trainloader, test=testloader)

class VGG16(nn.Module):
  def __init__(self):
    super().__init__()
    self.features = self._make_features()
    self.classification_head = nn.Linear(in_features=512, out_features=10)

  def forward(self, x):
    out = self.features(x)
    out = out.view(out.size(0), -1)
    out = self.classification_head(out)
    return out

  def _make_features(self):
    config = [64,64,'MP',128,128,'MP',256,256,256,'MP',512,512,512,'MP',512,512,512,'MP']
    layers = []
    c_in = 3
    for c in config:
      if c == 'MP':
        layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
      else:
        layers += [nn.Conv2d(in_channels=c_in, out_channels=c, kernel_size=3, stride=1, padding=1),
                   nn.BatchNorm2d(num_features=c),
                   nn.ReLU6(inplace=True)]
        c_in = c
    return nn.Sequential(*layers)

def imshow(images, labels, predicted, target_names):
  img = torchvision.utils.make_grid(images)
  plt.imshow(img.permute(1,2,0).cpu().numpy())
  [print(target_names[c], end=' ') for c in list(labels.cpu().numpy()) ]
  print()
  [print(target_names[c], end=' ') for c in list(predicted.cpu().numpy()) ]
  print()

def train_epoch(epoch, model, loader, loss_func, optimizer, device):
  model.train()
  running_loss = 0.0
  reporting_steps = 60
  for i, (images, labels) in enumerate(loader):
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    loss = loss_func(outputs, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    running_loss += loss.item()
    if i % reporting_steps == reporting_steps-1:
      print(f"Epoch {epoch} step {i} ave_loss {running_loss/reporting_steps:.4f}")
      running_loss = 0.0

def test_epoch(epoch, model, loader, device):
  ytrue = []
  ypred = []
  with torch.no_grad():
    model.eval()

    for i, (images, labels) in enumerate(loader):
      images, labels = images.to(device), labels.to(device)
      outputs = model(images)
      _, predicted = torch.max(outputs, dim=1)

      ytrue += list(labels.cpu().numpy())
      ypred += list(predicted.cpu().numpy())

  return ypred, ytrue

def main(PATH='./model.pth'):
  classes = get_classes()
  datasets = prepare_data()
  # img, label = datasets.train[0]
  # plt.imshow(img)
  # print(classes[label], img.size)
  # print('train', len(datasets.train), 'test', len(datasets.test))

  loaders = prepare_loader(datasets)
  # images, labels = iter(loaders.train).next()
  # print(images.shape, labels.shape)

  # Use GPU if available otherwise CPU
  device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
  print("Using device:", device)
  model = VGG16().to(device)
  # images, labels = iter(loaders.train).next()
  # outputs = model(images)
  # print(outputs.shape)
  # print(outputs[0])
  # _, predicted = torch.max(outputs, dim=1)
  # print(predicted)
  # imshow(images, labels, predicted, classes)

  loss_func = nn.CrossEntropyLoss()
  optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
  for epoch in range(10):
    train_epoch(epoch, model, loaders.train, loss_func, optimizer, device)
    ypred, ytrue = test_epoch(epoch, model, loaders.test, device)
    print(classification_report(ytrue, ypred, target_names=classes))

    torch.save(model.state_dict(), PATH)

  return model

model = main()

Using device: cpu
Epoch 0 step 59 ave_loss 2.1261
Epoch 0 step 119 ave_loss 1.7395
Epoch 0 step 179 ave_loss 1.5084
Epoch 0 step 239 ave_loss 1.4155
Epoch 0 step 299 ave_loss 1.2928
Epoch 0 step 359 ave_loss 1.2311
              precision    recall  f1-score   support

       plane       0.71      0.40      0.51      1000
         car       0.69      0.76      0.73      1000
        bird       0.46      0.24      0.31      1000
         cat       0.25      0.22      0.24      1000
        deer       0.37      0.40      0.38      1000
         dog       0.47      0.09      0.15      1000
        frog       0.25      0.94      0.40      1000
       horse       0.95      0.12      0.22      1000
        ship       0.59      0.82      0.69      1000
       truck       0.93      0.39      0.55      1000

    accuracy                           0.44     10000
   macro avg       0.57      0.44      0.42     10000
weighted avg       0.57      0.44      0.42     10000

Epoch 1 step 59 ave_loss 1

KeyboardInterrupt: 